# Welcome to AQUIFER

One lakehouse, three ways in. Trino, DuckDB and PyIceberg all read the same Iceberg tables in MinIO through the same catalog.
Run `./lakehouse demo` first so the `shop` tables exist. Run the cells with **Shift+Enter**.

In [ ]:
from aquifer_helpers import query, duck, catalog
import matplotlib.pyplot as plt

query("SHOW TABLES")

## 1. Trino: SQL into a DataFrame

In [ ]:
monthly = query('''
    SELECT date_trunc('month', order_date) AS month, round(sum(net_revenue)) AS revenue
    FROM daily_revenue
    WHERE order_date < date_trunc('month', current_date)
    GROUP BY 1 ORDER BY 1
''')
monthly.plot(x="month", y="revenue", kind="bar", legend=False, figsize=(9, 3.5), title="Net revenue by month (EUR)")
plt.tight_layout()
monthly.tail()

## 2. DuckDB: the same tables, in process

In [ ]:
con = duck()
con.sql('''
    SELECT country, round(sum(net_revenue)) AS revenue
    FROM lake.shop.daily_revenue
    GROUP BY 1 ORDER BY 2 DESC
''').df()

## 3. PyIceberg: tables as Arrow and pandas, plus metadata

In [ ]:
table = catalog().load_table("shop.customers")
print(table.schema())
table.scan(limit=5).to_pandas()

In [ ]:
[(snapshot.snapshot_id, snapshot.timestamp_ms) for snapshot in table.snapshots()][:3]

## 4. Streaming: Kafka to Flink to Iceberg

In a terminal run `./lakehouse stream --events 300 --rate 30`, wait about 20 seconds and run the cell again: the numbers move.

In [ ]:
query('''
    SELECT status, count(*) AS events, round(sum(amount)) AS order_value, max(ingested_at) AS last_ingested
    FROM order_events GROUP BY 1 ORDER BY 2 DESC
''')

## Your own data

`./lakehouse load my_file.csv` creates an Iceberg table from a CSV, TSV or JSON-lines file. Query it here with
`query("SELECT * FROM my_file LIMIT 10", schema="data")`.